In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../')
import random
from core import LADTransferTreeBoost, LSTransferTreeBoost, MTransferTreeBoost
from utils import *
import xgboost as xgb
from sklearn.model_selection import train_test_split
import itertools

from baselines import * #to import baseline methods

In [2]:
seed = 5209
seed_list = list(range(11,21))
predictor_columns = ['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad',
       'guestroom', 'hotwaterheating', 'airconditioning',
       'parking', 'furnishingstatus', 'basement']
target_column = 'price'


In [ ]:
data = pd.read_csv('../datasets/Housing.csv')

# Convert all object columns to categorical codes
for col in data.select_dtypes(include=['object']).columns:
    data[col] = data[col].astype('category').cat.codes

#choose a splitting variable
data_source = data[data['basement'] == 0]
data_target = data[data['basement'] == 1]


X_source_train, y_source_train = np.array(data_source[predictor_columns], dtype=float), np.array(data_source[target_column])

data_target_temp, data_target_test = train_test_split(data_target, test_size = 0.1, random_state=seed)
data_target_train, data_target_val = train_test_split(data_target_temp, test_size = 0.3, random_state=seed)

X_target_train, y_target_train = np.array(data_target_train[predictor_columns], dtype=float), np.array(data_target_train[target_column])
X_target_val, y_target_val = np.array(data_target_val[predictor_columns], dtype=float), np.array(data_target_val[target_column])
X_target_test, y_target_test = np.array(data_target_test[predictor_columns], dtype=float), np.array(data_target_test[target_column])

print(len(X_source_train), len(X_target_train), len(X_target_val), len(X_target_test))

In [ ]:
#Now test for xgboost, xgboost_naive and LSTransferTreeBoost

results = pd.DataFrame(columns = ['seed', 'method', 'v', 'target_tree_size', 'rmse', 'mae'])

v_list = [0.02, 0.05, 0.08, 0.1, 0.12, 0.15]
target_tree_size_list = [1,2,3,4]
combinations = list(itertools.product(v_list, target_tree_size_list))
for seed in seed_list:

    data = pd.read_csv('../datasets/Housing.csv')

    # Convert all object columns to categorical codes
    for col in data.select_dtypes(include=['object']).columns:
        data[col] = data[col].astype('category').cat.codes

    #choose a splitting variable
    data_source = data[data['prefarea'] == 0]
    data_target = data[data['prefarea'] == 1]


    X_source_train, y_source_train = np.array(data_source[predictor_columns], dtype=float), np.array(data_source[target_column])

    data_target_temp, data_target_test = train_test_split(data_target, test_size = 0.1, random_state=seed)
    data_target_train, data_target_val = train_test_split(data_target_temp, test_size = 0.3, random_state=seed)

    X_target_train, y_target_train = np.array(data_target_train[predictor_columns], dtype=float), np.array(data_target_train[target_column])
    X_target_val, y_target_val = np.array(data_target_val[predictor_columns], dtype=float), np.array(data_target_val[target_column])
    X_target_test, y_target_test = np.array(data_target_test[predictor_columns], dtype=float), np.array(data_target_test[target_column])
    X_comb_train = np.concatenate((X_target_train, X_source_train))
    y_comb_train = np.concatenate((y_target_train, y_source_train))

    print(len(X_source_train), len(X_target_train), len(X_target_val), len(X_target_test))
    
    for v, target_tree_size in combinations:
        params = {
            'objective': 'reg:squarederror',  # Regression with squared error
            'max_depth': target_tree_size,                   # Maximum depth of a tree
            'eta': v,                       # Learning rate
            'eval_metric': 'rmse',           # RMSE as evaluation metric
            }

        method = 'xgboost'       
        bst = train_xgboost(X_target_train, y_target_train, X_target_val, y_target_val, boosting_rounds=1000, params=params)
        preds = test_xgboost(X_target_test, bst)
        rmse = compute_rmse(preds, y_target_test)
        mae = compute_mae(preds, y_target_test)

        results.loc[len(results)] = [seed, method, v, target_tree_size, rmse, mae]

        method = 'naive_transfer'       
        bst = train_xgboost(X_comb_train, y_comb_train, X_target_val, y_target_val, boosting_rounds=1000, params=params)
        preds = test_xgboost(X_target_test, bst)
        rmse = compute_rmse(preds, y_target_test)
        mae = compute_mae(preds, y_target_test)

        results.loc[len(results)] = [seed, method, v, target_tree_size, rmse, mae]
        results.to_csv('results/xgboost_ablation.csv')




In [ ]:
#Now test for TwoStageTradaBoostR2

results = pd.DataFrame(columns = ['seed', 'method', 'lr', 'target_tree_size', 'rmse', 'mae'])
target_tree_size_list = [1]
lr_list = [1.0]
combinations = list(itertools.product(target_tree_size_list, lr_list))
for seed in seed_list:

    data = pd.read_csv('../datasets/Housing.csv')

    # Convert all object columns to categorical codes
    for col in data.select_dtypes(include=['object']).columns:
        data[col] = data[col].astype('category').cat.codes

    #choose a splitting variable
    data_source = data[data['prefarea'] == 0]
    data_target = data[data['prefarea'] == 1]


    X_source_train, y_source_train = np.array(data_source[predictor_columns], dtype=float), np.array(data_source[target_column])

    #Omit val data fro TradaBoost (at least for now)
    data_target_train, data_target_test = train_test_split(data_target, test_size = 0.1, random_state=seed)
    

    X_target_train, y_target_train = np.array(data_target_train[predictor_columns], dtype=float), np.array(data_target_train[target_column])
    X_target_val, y_target_val = np.array(data_target_val[predictor_columns], dtype=float), np.array(data_target_val[target_column])
    X_target_test, y_target_test = np.array(data_target_test[predictor_columns], dtype=float), np.array(data_target_test[target_column])
    X_comb_train = np.concatenate((X_target_train, X_source_train))
    y_comb_train = np.concatenate((y_target_train, y_source_train))

    print(len(X_source_train), len(X_target_train), len(X_target_val), len(X_target_test))
    
    for target_tree_size, lr in combinations:

        method = 'twostagetradaboostr2'       
        model = train_twostagetradaboostr2(X_source_train, y_source_train, X_target_train, y_target_train, n_estimators=1000, 
                                           tree_size_depth=target_tree_size,lr=lr)
        preds = predict_twostagetradaboostr2(X_target_test, model)
        rmse = compute_rmse(preds, y_target_test)
        mae = compute_mae(preds, y_target_test)

        results.loc[len(results)] = [seed, method, lr, target_tree_size, rmse, mae]
        results.to_csv('results/tradaboostr2_ablation.csv')



417 80 35 13
417 80 35 13


c:\Users\Dag Bjornberg\AI\transfertreeboost\env\Lib\site-packages\adapt\instance_based\_tradaboost.py:541: SyntaxWarning: invalid escape sequence '\c'
  max_{\\epsilon \\in \\epsilon_S \cup \\epsilon_T} \\epsilon`.
c:\Users\Dag Bjornberg\AI\transfertreeboost\env\Lib\site-packages\adapt\instance_based\_tradaboost.py:700: SyntaxWarning: invalid escape sequence '\c'
  max_{\\epsilon \\in \\epsilon_S \cup \\epsilon_T} \\epsilon`.


KeyboardInterrupt: 

In [ ]:
results = pd.DataFrame(columns = ['seed', 'method', 'v', 'source_tree_size', 'target_tree_size', 'k', 'm_0', 'rmse', 'mae'])

v_list = [0.05, 0.1]
source_tree_size_list = [1,2]
target_tree_size_list = [1,2]
k_list = [0.05]
m_0_list = [0.5, 0.9]
combinations = list(itertools.product(v_list, source_tree_size_list, target_tree_size_list, 
                                     k_list, m_0_list))
for seed in seed_list:

    data = pd.read_csv('../datasets/Housing.csv')

    # Convert all object columns to categorical codes
    for col in data.select_dtypes(include=['object']).columns:
        data[col] = data[col].astype('category').cat.codes

    #choose a splitting variable
    data_source = data[data['prefarea'] == 0]
    data_target = data[data['prefarea'] == 1]


    X_source_train, y_source_train = np.array(data_source[predictor_columns], dtype=float), np.array(data_source[target_column])

    data_target_temp, data_target_test = train_test_split(data_target, test_size = 0.1, random_state=seed)
    data_target_train, data_target_val = train_test_split(data_target_temp, test_size = 0.3, random_state=seed)

    X_target_train, y_target_train = np.array(data_target_train[predictor_columns], dtype=float), np.array(data_target_train[target_column])
    X_target_val, y_target_val = np.array(data_target_val[predictor_columns], dtype=float), np.array(data_target_val[target_column])
    X_target_test, y_target_test = np.array(data_target_test[predictor_columns], dtype=float), np.array(data_target_test[target_column])

    for v, source_tree_size, target_tree_size, k, m_0 in combinations:
        method = f'LSTransferTreeBoost'
        fiter = LSTransferTreeBoost(epochs=1000, v=v, source_tree_size=source_tree_size, 
                                    target_tree_size=target_tree_size, alpha_0=1.0, k=k, m_0=m_0)
        fiter.fit(X_target_train, y_target_train, X_source_train, y_source_train, val_x=X_target_val, val_y=y_target_val, early_stopping_rounds=8)
        rmse = fiter.evaluate(X_target_test, y_target_test, metric = 'rmse')
        mae = fiter.evaluate(X_target_test, y_target_test, metric = 'mae')
        results.loc[len(results)] = [seed, method, v, source_tree_size, target_tree_size, k, m_0, rmse, mae]
        results.to_csv('results/LSTransferTreeBoost_ablation.csv')


In [ ]:
#Find best hyperparameter values
dat_abl = pd.read_csv('results/xgboost_ablation.csv')
dat_abl_xgboost = dat_abl[dat_abl['method']=='xgboost']
dat_abl_naive = dat_abl[dat_abl['method']=='naive_transfer']
dat_abl_xgboost = dat_abl_xgboost.sort_values(by = ['target_tree_size', 'v']).reset_index(drop=True)
group_index = dat_abl_xgboost.index // len(seed_list)

# Group by the group index and sum, then broadcast to original rows
dat_abl_xgboost['total_rmse'] = dat_abl_xgboost.groupby(group_index)['rmse'].transform('sum') / len(seed_list) #or how many seeds we have used!
dat_abl_xgboost['total_mae'] = dat_abl_xgboost.groupby(group_index)['mae'].transform('sum') / len(seed_list) #or how many seeds we have used!
dat_abl_xgboost= dat_abl_xgboost.sort_values(by = ['total_rmse']) #select on total_mae or total_rmse
#now, fin best parameters for each disturbance_level (we choose top 3)
optimal_params_transfer = pd.DataFrame(columns = ['target_tree_size', 'v', 'total_rmse', 'total_mae'])
best_params = dat_abl_xgboost[['target_tree_size', 'v', 'total_rmse', 'total_mae']].iloc[[0, len(seed_list), len(seed_list)*2]]
optimal_params_transfer = pd.concat((optimal_params_transfer, best_params))

optimal_params_transfer['total_rmse'] = np.round(optimal_params_transfer['total_rmse'], 2)
optimal_params_transfer['total_mae'] = np.round(optimal_params_transfer['total_mae'], 2)
optimal_params_transfer


In [ ]:
dat_abl_naive = dat_abl_naive.sort_values(by = ['target_tree_size', 'v']).reset_index(drop=True)
group_index = dat_abl_naive.index // len(seed_list)

# Group by the group index and sum, then broadcast to original rows
dat_abl_naive['total_rmse'] = dat_abl_naive.groupby(group_index)['rmse'].transform('sum') / len(seed_list) #or how many seeds we have used!
dat_abl_naive['total_mae'] = dat_abl_naive.groupby(group_index)['mae'].transform('sum') / len(seed_list) #or how many seeds we have used!
dat_abl_naive= dat_abl_naive.sort_values(by = ['total_rmse']) #select on total_mae or total_rmse
#now, fin best parameters for each disturbance_level (we choose top 3)
optimal_params_transfer = pd.DataFrame(columns = ['target_tree_size', 'v', 'total_rmse', 'total_mae'])
best_params = dat_abl_naive[['target_tree_size', 'v', 'total_rmse', 'total_mae']].iloc[[0, len(seed_list), len(seed_list)*2]]
optimal_params_transfer = pd.concat((optimal_params_transfer, best_params))

optimal_params_transfer['total_rmse'] = np.round(optimal_params_transfer['total_rmse'], 2)
optimal_params_transfer['total_mae'] = np.round(optimal_params_transfer['total_mae'], 2)
optimal_params_transfer

In [13]:
dat_abl_tradaboostr2 = pd.read_csv('results/tradaboostr2_ablation.csv')
dat_abl_tradaboostr2 = dat_abl_tradaboostr2.sort_values(by = ['lr', 'target_tree_size']).reset_index(drop=True)
group_index = dat_abl_tradaboostr2.index // len(seed_list)

# Group by the group index and sum, then broadcast to original rows
dat_abl_tradaboostr2['total_rmse'] = dat_abl_tradaboostr2.groupby(group_index)['rmse'].transform('sum') / len(seed_list) #or how many seeds we have used!
dat_abl_tradaboostr2['total_mae'] = dat_abl_tradaboostr2.groupby(group_index)['mae'].transform('sum') / len(seed_list) #or how many seeds we have used!
dat_abl_tradaboostr2= dat_abl_tradaboostr2.sort_values(by = ['total_rmse']) #select on total_mae or total_rmse
#now, fin best parameters for each disturbance_level (we choose top 3)
optimal_params_transfer = pd.DataFrame(columns = ['lr', 'target_tree_size', 'total_rmse', 'total_mae'])
best_params = dat_abl_tradaboostr2[['lr', 'target_tree_size', 'total_rmse', 'total_mae']].iloc[[0, len(seed_list), len(seed_list)*2]]
optimal_params_transfer = pd.concat((optimal_params_transfer, best_params))

optimal_params_transfer['total_rmse'] = np.round(optimal_params_transfer['total_rmse'], 2)
optimal_params_transfer['total_mae'] = np.round(optimal_params_transfer['total_mae'], 2)
optimal_params_transfer

IndexError: positional indexers are out-of-bounds

In [ ]:
#Find best hyperparameter values
dat_abl = pd.read_csv('results/LSTransferTreeBoost_ablation.csv')
dat_abl = dat_abl.sort_values(by = ['v', 'source_tree_size', 'target_tree_size', 'k', 'm_0']).reset_index(drop=True)
group_index = dat_abl.index // len(seed_list)

# Group by the group index and sum, then broadcast to original rows
dat_abl['total_rmse'] = dat_abl.groupby(group_index)['rmse'].transform('sum') / len(seed_list) #or how many seeds we have used!
dat_abl['total_mae'] = dat_abl.groupby(group_index)['mae'].transform('sum') / len(seed_list) #or how many seeds we have used!
dat_abl= dat_abl.sort_values(by = ['total_rmse']) #select on total_mae or total_rmse
#now, fin best parameters for each disturbance_level (we choose top 3)
optimal_params_transfer = pd.DataFrame(columns = ['v', 'source_tree_size', 'target_tree_size', 'k', 'm_0', 'total_rmse', 'total_mae'])
best_params = dat_abl[['v', 'source_tree_size', 'target_tree_size', 'k', 'm_0', 'total_rmse', 'total_mae']].iloc[[0, len(seed_list), len(seed_list)*2]]
optimal_params_transfer = pd.concat((optimal_params_transfer, best_params))

optimal_params_transfer['total_rmse'] = np.round(optimal_params_transfer['total_rmse'], 2)
optimal_params_transfer['total_mae'] = np.round(optimal_params_transfer['total_mae'], 2)
optimal_params_transfer